In [128]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import root_mean_squared_error, make_scorer # accuracy for classification task.
from sklearn.datasets import make_regression

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import BaggingRegressor

import pandas as pd

#                                                Theory
```js
        Everything is same as DecisionTreeClassifier, but here each model in bagging returns a continuous value and BaggingRegressor returns their average.
```

#                                                Dataset Creation

In [50]:
df = pd.read_csv(r"D:\VS CODE\Data Science\Machine Learning\7. Bagging\Dataset\HousingData.csv").fillna(0)

X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values
print(f"X.shape, y.shape = {X.shape}, {y.shape}.\n")

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(f"x_train.shape, x_test.shape, y_train.shape, y_test.shape = {x_train.shape, x_test.shape, y_train.shape, y_test.shape}.")

X.shape, y.shape = (506, 13), (506,).

x_train.shape, x_test.shape, y_train.shape, y_test.shape = ((404, 13), (102, 13), (404,), (102,)).


#                                               Comparing all model's Errors

In [106]:
lr  = LinearRegression().fit(X= x_train, y = y_train)
dt  = DecisionTreeRegressor(max_depth = None, random_state = 42).fit(X= x_train, y = y_train)
knn = KNeighborsRegressor().fit(X= x_train, y = y_train)
bag = BaggingRegressor(estimator = DecisionTreeRegressor(),
                       n_estimators = 15, max_samples = 0.4, n_jobs = -1, random_state = 42).fit(X= x_train, y = y_train)

print("                      Prediction on Training Dataset :\n")
print(f"RMSE for Linear = {root_mean_squared_error(y_true = y_train, y_pred = lr.predict(x_train))}.")
print(f"RMSE for Decision Tree = {root_mean_squared_error(y_true = y_train, y_pred = dt.predict(x_train))}.")
print(f"RMSE for KNN = {root_mean_squared_error(y_true = y_train, y_pred = knn.predict(x_train))}.")
print(f"RMSE for Bagging = {root_mean_squared_error(y_true = y_train, y_pred = bag.predict(x_train))}.")

print("\n                      Prediction on Testing Dataset :\n")
print(f"RMSE for Linear = {root_mean_squared_error(y_true = y_test, y_pred = lr.predict(x_test))}.")
print(f"RMSE for Decision Tree = {root_mean_squared_error(y_true = y_test, y_pred = dt.predict(x_test))}.")
print(f"RMSE for KNN = {root_mean_squared_error(y_true = y_test, y_pred = knn.predict(x_test))}.")
print(f"RMSE for Bagging = {root_mean_squared_error(y_true = y_test, y_pred = bag.predict(x_test))}.")

explanation = """
Even though Decision Tree performed 100% well in training dataset, in testing it performed poor with high errors i.e.
high variance indicating overfitting. But bagging performed well in both training and testing with lower difference of
their errors indicating low bias obviously and low variance.
"""

                      Prediction on Training Dataset :

RMSE for Linear = 4.762783394728861.
RMSE for Decision Tree = 0.0.
RMSE for KNN = 5.66728461006549.
RMSE for Bagging = 2.7804513250472653.

                      Prediction on Testing Dataset :

RMSE for Linear = 5.09212102660837.
RMSE for Decision Tree = 4.969297894937217.
RMSE for KNN = 5.895657092217933.
RMSE for Bagging = 3.1079465230942986.


#                                               Applying GridSearchCV

In [137]:
parameters = {
                'estimator'    : [DecisionTreeRegressor(), LinearRegression(), KNeighborsRegressor()],
                'n_estimators' : [5, 10, 15, 20],
                'max_samples'  : [0.25, 0.4, 0.5],
                'max_features' : [0.5, 0.6, 1],
                'bootstrap'    : [True, False],
                'bootstrap_features' : [True, False],
                'random_state' : [42, 50, 60]
}

search = GridSearchCV(estimator = BaggingRegressor(), param_grid = parameters, cv = 5, n_jobs = -1,
                      scoring = "neg_root_mean_squared_error", # by default best_score_ returns r2 score.
                      refit = "neg_root_mean_squared_error") # when we'll predict, which score we want to see.

search.fit(X = x_train, y = y_train)

print(f"Best RMSE score : {-search.best_score_}.\n")
print(f"Best parameters :\n {search.best_params_}.")

Best RMSE score : 4.129839134037873.

Best parameters :
 {'bootstrap': False, 'bootstrap_features': False, 'estimator': DecisionTreeRegressor(), 'max_features': 0.6, 'max_samples': 0.4, 'n_estimators': 20, 'random_state': 50}.


In [140]:
bag1 = BaggingRegressor(estimator = DecisionTreeRegressor(),
                        n_estimators = 20,
                        max_samples = 0.4,
                        max_features = 0.6,
                        bootstrap = False,
                        bootstrap_features = False,
                        n_jobs = -1,
                        random_state = 50)

bag1.fit(X= x_train, y = y_train)

print("                      Prediction on Training Dataset :")
print(f"RMSE for Bagging = {root_mean_squared_error(y_true = y_train, y_pred = bag1.predict(x_train))}.")

print("\n                      Prediction on Testing Dataset :")
print(f"RMSE for Bagging = {root_mean_squared_error(y_true = y_test, y_pred = bag1.predict(x_test))}.")

i_am_confused = """
Above we saw :
                      Prediction on Training Dataset :
    RMSE for Bagging = 2.7804513250472653.

                      Prediction on Testing Dataset :
    RMSE for Bagging = 3.1079465230942986.

But Here :
                      Prediction on Training Dataset :
    RMSE for Bagging = 2.8282429034282184.

                      Prediction on Testing Dataset :
    RMSE for Bagging = 3.792796771900807.
"""

                      Prediction on Training Dataset :
RMSE for Bagging = 2.8282429034282184.

                      Prediction on Testing Dataset :
RMSE for Bagging = 3.792796771900807.
